In [1]:
!pip install -q --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "datasets<4.4.0" "trl<=0.24.0"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 91.8 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 293.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 80.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 75.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 350.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 74.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets =UserSecretsClient()
    hf_token=user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("logged in the hugging face account using the token")
except Exception as e:
    print("Not able to log in")

In [ ]:
from unsloth import FastLanguageModel # to import the model (mistral)
from datasets import load_dataset  # to import our dataset from  hf
from unsloth.chat_templates import get_chat_template # this is used to tokenise the message
# chatml is used like a set of rules which helps the llm to understand which part of message is the query by user and 
# and which part is the solution to the query using delimiter <start>system<end><start>user:....<end><start>user:<end>
# basically a tokeniser but not fully tokeninsing our message

In [ ]:
# calling the specefic llm
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=4096,
    # max_seq_length=4096 means we have 4096 tokens (ie 3k words can be handled).token!=word btw.
    
    dtype=None,
    # datatype is none as we don't concern ourselves with the datatype of the dataset (auto handled)
    
    load_in_4bit=True,# we are calling the 4bit instead of the std. 16bits as it will be correct on the kaggle nb
    # otherwise the kaggle will not be able to handle the 16 bit gpu memory processing
)

In [ ]:
tokenizer=get_chat_template(
    tokenizer,
    chat_template="chatml",
)

In [ ]:
def format_for_ai(row):
    row["text"]=tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return row
print("loading the dataset we created from hugging face\n")
dataset_path="GAnirudh711/DSA_FINETUNING_LLM"
mega_dataset=load_dataset(dataset_path,split="train")

print("converting the dataset in the required format using map\n")
mega_dataset=mega_dataset.map(format_for_ai,num_proc=2)
print("converted the initial dataet in the correct format\n")
print(f'the length is {len(mega_dataset)}')


In [ ]:
model=FastLanguageModel.get_peft_model(
    model,
    r=16,# r is the rank of the matrix which will be used for the d*r and r*k 

    target_modules=[
        "q_proj","k_proj", # part of attention 
        "v_proj","o_proj", # part of attention
        "gate_proj", # part of mlp
        "up_proj","down_proj" # part of mlp
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42
)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
total=model.num_parameters()
frozen=total-trainable

print(f"Lora adapters attached")
print(f"Total parameters:{total:>14,}")
print(f"Trainable (sticky notes):{trainable:>10,})({100*trainable/total:.2f}%)")
print(f"Frozen(main brain):{frozen:>14,}({100*frozen/total:.2f}%)")


    
    


In [ ]:
# ==========================================
# PHASE 2: TRAINING (CELL 4)
# The Training Loop
# ==========================================

from trl import SFTTrainer
from transformers import TrainingArguments
import torch

print(" Setting up the Trainer...")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=mega_dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    dataset_num_proc=2,
    packing=False,

    args=TrainingArguments(
        # --- Batch size & Memory ---
        per_device_train_batch_size=1,       # 1 question at a time (keeps 4096 tokens safe in VRAM)
        gradient_accumulation_steps=8,       # Update weights after every 8 questions

        # --- How long to train ---
        max_steps=500,                       # ~1.5 to 2 hours on Kaggle T4
        warmup_steps=10,                     # Gently ramp up learning rate at the start

        # --- Learning settings ---
        learning_rate=2e-4,                  # The optimal learning rate for LoRA
        lr_scheduler_type="cosine",          # Gradually slows down learning as it gets smarter
        weight_decay=0.01,
        optim="adamw_8bit",                  # 8-bit optimizer to save memory

        # --- Precision & Logging ---
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,                    # Print loss progress every 10 steps
        output_dir="outputs",
        report_to="none",                    # No external logging dashboards needed
        average_tokens_across_devices=False, # Fixes the recent Hugging Face version mismatch
        seed=42,
    ),
)
print("\n STARTING TRAINING!")
print("Keep an eye on the 'Loss' column — it should steadily decrease.\n")
trainer_stats = trainer.train()
print("\n TRAINING COMPLETE!")

In [ ]:
# ==========================================
# TEST YOUR TRAINED MODEL
# ==========================================

from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer

checkpoint_path = "outputs/checkpoint-500"
print(f" Loading your trained weights from {checkpoint_path}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=checkpoint_path,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

# Switch to fast inference
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = "You are an expert DSA (Data Structures & Algorithms) tutor and competitive programmer. Explain your approach and provide clean code."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            "Solve the Maximum Subarray problem (Kadane's Algorithm). "
            "Write the solution strictly in modern C++ with comments, "
            "and explain the Time and Space complexity."
        ),
    },
]

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    add_generation_prompt=True, 
    return_tensors="pt"
).to("cuda")

print("\n🧠 Generating solution live...\n" + "=" * 50)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=1024,
    temperature=0.3,
    use_cache=True,
)

In [ ]:
# ==========================================
# DIRECT CLOUD BACKUP (NO UNSLOTH NEEDED!)
# ==========================================

from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient
import os

# 1. Login to Hugging Face
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# 2. Check that your checkpoint folder exists on disk
checkpoint_path = "outputs/checkpoint-500"

if os.path.exists(checkpoint_path):
    print(f" Found checkpoint on disk! Files: {os.listdir(checkpoint_path)}")
    
    # 3. Upload the entire folder directly to Hugging Face
    api = HfApi()
    repo_id = "GAnirudh711/mistral-7b-dsa-expert"
    
    print(f"\n Uploading directly to https://huggingface.co/{repo_id}...")
    api.upload_folder(
        folder_path=checkpoint_path,
        repo_id=repo_id,
        repo_type="model",
    )
    
    print("\n MODEL IS PERMANENTLY SAVED TO THE CLOUD!")
    print(f"Go view it live: https://huggingface.co/{repo_id}")
else:
    print(f" Could not find {checkpoint_path}. Current folder contains:", os.listdir("."))

In [ ]:
# ==========================================
# TEST: FULL TUTOR MODE (CONCEPT + INTUITION + CODE)
# ==========================================

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from transformers import TextStreamer

tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "You are a Senior Algorithms Instructor and Technical Interview Coach. "
    "Your goal is to make students deeply understand the concepts. "
    "Always structure your answer into: \n"
    "1. High-Level Concept & Intuition (explain the core idea simply)\n"
    "2. Algorithmic Breakdown (why this approach works and key observations)\n"
    "3. Time and Space Complexity analysis\n"
    "4. Production-Grade C++ Solution with thorough comments"
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {
        "role": "user",
        "content": (
            "Explain and solve the 'Trapping Rain Water' problem.\n"
            "I want to understand the intuition behind the Two-Pointer approach first—specifically, "
            "why do we always advance the pointer with the smaller height? "
            "Then break down the complexity and provide the optimal modern C++ code."
        ),
    },
]

inputs = tokenizer.apply_chat_template(
    messages, 
    tokenize=True, 
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

print("\n🎓 TUTOR IN ACTION (Intuition + Code)...\n" + "=" * 50)
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=1500, # Increased so it has plenty of room for both explanation and code!
    temperature=0.4,     # Slightly higher so it explains things naturally
    use_cache=True,
)